# Role 2 – Baseline Models (Elliptic Bitcoin)

Run the notebook **top to bottom** in Google Colab. It downloads the public Elliptic dataset from KaggleHub, reads the team split from `MyDrive/ML-project-main/tests/elliptic_split.csv`, and uses the evaluation module from that same Drive folder. Keep the folder structure unchanged.

**Methods:** Train 1–29 / validation 30–34 for model selection, then retrain the selected random forest on labelled periods 1–34 and evaluate on labelled periods 35–49. The graph degree is calculated from same-period edges; this is a batch-graph assumption, not proof of online availability.

**Outputs:** The final prediction CSV and four evaluation files are copied to `MyDrive/Role2_Results`. Each evaluator run uses a fresh temporary output directory so rerunning does not overwrite a previous evaluator run.

**Review note:** Outputs in this shared copy are cleared; results must be reproduced by running the notebook. The final reported evaluation is subject to confirming permitted feature availability and graph information rules with the team. Do not commit private tokens or local datasets.


In [ ]:

%pip -q install kagglehub

import kagglehub
from pathlib import Path

dataset_path = kagglehub.dataset_download(
    "ellipticco/elliptic-data-set"
)

print("Dataset location:", dataset_path)

for file in Path(dataset_path).rglob("*"):
    if file.is_file():
        print(file.relative_to(dataset_path))


In [ ]:

import pandas as pd
from pathlib import Path

folder = Path(dataset_path) / "elliptic_bitcoin_dataset"

features = pd.read_csv(
    folder / "elliptic_txs_features.csv",
    header=None
)

classes = pd.read_csv(
    folder / "elliptic_txs_classes.csv"
)

edges = pd.read_csv(
    folder / "elliptic_txs_edgelist.csv"
)

print("Features:", features.shape)
print("Classes:", classes.shape)
print("Edges:", edges.shape)

print("\nFirst 5 transactions:")
display(features.head())

print("\nClass distribution:")
print(classes["class"].value_counts())


In [ ]:


import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

split_path = Path(
    "/content/drive/MyDrive/ML-project-main/tests/elliptic_split.csv"
)

assert split_path.exists(), "Split file not found"

split = pd.read_csv(
    split_path,
    dtype={"txId": str, "class": str}
)

assert len(split) == 203769
assert split["txId"].is_unique

print("SPLIT LOADED SUCCESSFULLY")
print("Total rows:", len(split))
print(split["split"].value_counts())


In [ ]:

import numpy as np
import pandas as pd

# Prepare original Elliptic features
feature_data = features.copy()

feature_data.columns = (
    ["txId", "time_step"] +
    [f"feature_{i}" for i in range(1, 166)]
)

feature_data["txId"] = feature_data["txId"].astype(str)

# Join with the official team split
data = feature_data.merge(
    split,
    on=["txId", "time_step"],
    how="inner",
    validate="one_to_one"
)

assert len(data) == len(split), "Some transactions are missing"

# Use the 165 anonymized numerical features
feature_columns = [
    f"feature_{i}" for i in range(1, 166)
]

# Keep only transactions with known labels
labelled = data[data["class"].isin(["licit", "illicit"])].copy()
labelled["target"] = (labelled["class"] == "illicit").astype(int)

# Development: train 1-29, validation 30-34
train = labelled[
    (labelled["split"] == "train") &
    (labelled["time_step"] <= 29)
].copy()

validation = labelled[
    (labelled["split"] == "train") &
    (labelled["time_step"].between(30, 34))
].copy()

test = labelled[
    labelled["split"] == "test"
].copy()

print("Training:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))
print("Features:", len(feature_columns))

print("\nTraining classes:")
print(train["class"].value_counts())


In [ ]:

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Create the training and validation inputs
X_train = train[feature_columns]
y_train = train["target"]

X_val = validation[feature_columns]
y_val = validation["target"]

# Build the baseline model
logistic_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=0
    )
)

# Train only on the training period
logistic_model.fit(X_train, y_train)

# Predict illicit probabilities on validation data
val_scores = logistic_model.predict_proba(X_val)[:, 1]
val_predictions = (val_scores >= 0.50).astype(int)

print("LOGISTIC REGRESSION - VALIDATION")
print("Precision:", round(precision_score(y_val, val_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, val_predictions, zero_division=0), 4))
print("F1:", round(f1_score(y_val, val_predictions, zero_division=0), 4))
print("Average Precision:", round(average_precision_score(y_val, val_scores), 4))



In [ ]:

%pip -q install xgboost

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score
)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        class_weight="balanced",
        random_state=0,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=0,
        n_jobs=2
    )
}

validation_results = []

# Include our existing Logistic Regression
validation_results.append({
    "Model": "Logistic Regression",
    "Precision": precision_score(y_val, val_predictions),
    "Recall": recall_score(y_val, val_predictions),
    "F1": f1_score(y_val, val_predictions),
    "AP": average_precision_score(y_val, val_scores)
})

for name, model in models.items():
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_val)[:, 1]
    predictions = (scores >= 0.50).astype(int)

    validation_results.append({
        "Model": name,
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "AP": average_precision_score(y_val, scores)
    })

results_df = pd.DataFrame(validation_results)
display(results_df.round(4))


In [ ]:

# Graph feature: within-period degree
node_periods = feature_data[["txId", "time_step"]].copy()

edge_pairs = edges.iloc[:, :2].copy()
edge_pairs.columns = ["source", "target"]
edge_pairs = edge_pairs.astype(str)

edge_pairs = edge_pairs.merge(
    node_periods.rename(columns={
        "txId": "source",
        "time_step": "source_period"
    }),
    on="source",
    how="left",
    validate="many_to_one"
)

edge_pairs = edge_pairs.merge(
    node_periods.rename(columns={
        "txId": "target",
        "time_step": "target_period"
    }),
    on="target",
    how="left",
    validate="many_to_one"
)

assert edge_pairs[
    ["source_period", "target_period"]
].notna().all().all()

# Keep edges within the same time period
same_period = edge_pairs[
    edge_pairs["source_period"] ==
    edge_pairs["target_period"]
]

all_endpoints = pd.concat([
    same_period["source"],
    same_period["target"]
])

degree = all_endpoints.value_counts()

data["graph_degree"] = (
    data["txId"].map(degree).fillna(0).astype(int)
)

print("Graph feature created successfully!")
print("Edges in the same period:", len(same_period))
print("Maximum degree:", data["graph_degree"].max())
print("Average degree:", round(data["graph_degree"].mean(), 4))


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score,
    f1_score, average_precision_score
)

# Add the graph feature to our 165 original features
graph_features = feature_columns + ["graph_degree"]

# Prepare training and validation data with graph information
graph_train = data.loc[train.index]
graph_val = data.loc[validation.index]

X_train_graph = graph_train[graph_features]
X_val_graph = graph_val[graph_features]

# Train the same Random Forest configuration
graph_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=0,
    n_jobs=-1
)

graph_model.fit(X_train_graph, y_train)

graph_scores = graph_model.predict_proba(X_val_graph)[:, 1]
graph_predictions = (graph_scores >= 0.50).astype(int)

graph_result = pd.DataFrame([{
    "Model": "Random Forest + Degree",
    "Precision": precision_score(y_val, graph_predictions, zero_division=0),
    "Recall": recall_score(y_val, graph_predictions, zero_division=0),
    "F1": f1_score(y_val, graph_predictions, zero_division=0),
    "AP": average_precision_score(y_val, graph_scores)
}])

comparison = pd.concat([
    results_df[results_df["Model"] == "Random Forest"],
    graph_result
], ignore_index=True)

display(comparison.round(4))


In [ ]:

# Verify the chronological split
assert train["time_step"].between(1, 29).all()
assert validation["time_step"].between(30, 34).all()
assert test["time_step"].between(35, 49).all()

# Verify there are no overlapping transactions
train_ids = set(train["txId"])
val_ids = set(validation["txId"])
test_ids = set(test["txId"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

# Verify all model inputs are numerical
assert all(
    pd.api.types.is_numeric_dtype(data[col])
    for col in feature_columns + ["graph_degree"]
)

# Verify that all expected labelled rows are present
assert len(train) + len(validation) == 29894
assert len(test) == 16670

# Check the feature schema and graph periods
assert len(feature_columns) == 165
assert (edge_pairs["source_period"] ==
        edge_pairs["target_period"]).all()

print("ALL STRUCTURAL CHECKS PASSED")
print("Training periods: 1-29")
print("Validation periods: 30-34")
print("Test periods: 35-49")
print("No overlapping transaction IDs")
print("All 165 numerical features verified")
print("Graph edges stay within the same period")


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, average_precision_score

# All labelled training transactions, periods 1-34
final_train = labelled[
    labelled["split"] == "train"
].copy()

assert final_train["time_step"].between(1, 34).all()
assert test["time_step"].between(35, 49).all()

X_final_train = final_train[feature_columns]
y_final_train = final_train["target"]

X_final_test = test[feature_columns]

final_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=0,
    n_jobs=-1
)

final_model.fit(X_final_train, y_final_train)

# Export probabilities, not predicted labels
illicit_index = list(final_model.classes_).index(1)
test_scores = final_model.predict_proba(
    X_final_test
)[:, illicit_index]

predictions_df = pd.DataFrame({
    "txId": test["txId"].astype(str).values,
    "score_illicit": test_scores
})

assert len(predictions_df) == 16670
assert predictions_df["txId"].is_unique
assert predictions_df["score_illicit"].between(0, 1).all()

predictions_df.to_csv(
    "random_forest_seed0_predictions.csv",
    index=False
)

print("FINAL MODEL TRAINED SUCCESSFULLY")
print("Training examples:", len(final_train))
print("Test predictions:", len(predictions_df))
print("Prediction file saved successfully")
display(predictions_df.head())


In [ ]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

# These labels are used only for evaluation
y_test = test["target"].to_numpy()

test_predictions = (test_scores >= 0.50).astype(int)

precision = precision_score(
    y_test, test_predictions, zero_division=0
)
recall = recall_score(
    y_test, test_predictions, zero_division=0
)
f1 = f1_score(
    y_test, test_predictions, zero_division=0
)
ap = average_precision_score(y_test, test_scores)

tn, fp, fn, tp = confusion_matrix(
    y_test, test_predictions, labels=[0, 1]
).ravel()

print("FINAL RANDOM FOREST - TEST RESULTS")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))
print("Average Precision:", round(ap, 4))
print()
print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)


In [ ]:

from pathlib import Path

# Expected known-label test transaction IDs
expected_ids = set(
    split.loc[
        (split["split"] == "test") &
        (split["class"].isin(["licit", "illicit"])),
        "txId"
    ].astype(str)
)

# Actual prediction IDs
actual_ids = set(
    predictions_df["txId"].astype(str)
)

assert len(expected_ids) == 16670
assert len(actual_ids) == 16670
assert expected_ids == actual_ids
assert predictions_df["txId"].is_unique
assert predictions_df["score_illicit"].notna().all()
assert predictions_df["score_illicit"].between(0, 1).all()

output_file = Path("random_forest_seed0_predictions.csv")
assert output_file.exists()

print("PREDICTION FILE VALIDATION PASSED")
print("All 16,670 IDs match the shared test split")
print("No duplicate or missing transaction IDs")
print("All scores are between 0 and 1")
print("Ready for Role 4 evaluation")


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

repo = Path("/content/drive/MyDrive/ML-project-main")

print("Folder found:", repo.exists())
print(
    "Evaluator found:",
    (repo / "src" / "role4_evaluation.py").exists()
)
print(
    "Split found:",
    (repo / "tests" / "elliptic_split.csv").exists()
)


In [ ]:

import subprocess
import sys
from pathlib import Path

# Use the repository from Google Drive
repo = Path("/content/drive/MyDrive/ML-project-main")

# Copy predictions into Colab's temporary storage
prediction_file = Path("/content/random_forest_seed0_predictions.csv")

assert prediction_file.exists(), "Prediction file not found"
assert (repo / "src" / "role4_evaluation.py").exists()

# Install the team's evaluation dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-q", "-r", str(repo / "requirements-evaluation.txt")
    ],
    check=True
)

# Give each evaluation a fresh output folder (the evaluator never overwrites runs).
import tempfile
evaluation_dir = Path(tempfile.mkdtemp(prefix="role2_eval_")) / "results"

# Run Giulio's official evaluator
command = [
    sys.executable, "-m", "src.role4_evaluation",
    "--split-file", "tests/elliptic_split.csv",
    "--predictions", str(prediction_file),
    "--model", "RandomForest",
    "--seed", "0",
    "--threshold", "0.50",
    "--outdir", str(evaluation_dir)
]

subprocess.run(command, cwd=str(repo), check=True)

print("OFFICIAL EVALUATION COMPLETED")


In [ ]:

from pathlib import Path
import shutil
import pandas as pd

source = evaluation_dir
destination = Path(
    "/content/drive/MyDrive/Role2_Results"
)

destination.mkdir(parents=True, exist_ok=True)

# Save prediction CSV
shutil.copy2(
    "/content/random_forest_seed0_predictions.csv",
    destination / "random_forest_seed0_predictions.csv"
)

# Save the four official evaluation files
expected_files = [
    "overall_metrics.csv",
    "metrics_per_time_step.csv",
    "evaluated_predictions.csv",
    "run_metadata.json"
]

for filename in expected_files:
    source_file = source / filename
    assert source_file.exists(), f"Missing: {filename}"
    shutil.copy2(source_file, destination / filename)

print("ALL FILES SAVED TO GOOGLE DRIVE")
print("Location:", destination)

print("\nOFFICIAL TEST METRICS:")
display(pd.read_csv(destination / "overall_metrics.csv").T)
